# Colab: real-data PINN, batch 5: M8 experiments

12 runs, about 4 hours. They answer three questions about the zoned model: does the physics help, does inferred pumping help, and how many zones are best.
Training scores validation only; the test wells stay sealed.

**Do not change any cell.** Just follow these steps:
1. Put the **new** `pinn_code.zip` (from the project lead) in a Google Drive folder called `pinn`, so you have `MyDrive/pinn/pinn_code.zip`. Replace any older copy.
2. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Allow Google Drive access when asked.
3. Keep the laptop awake (plugged in, sleep off, this tab open). If Colab disconnects, just **Run all** again: finished runs are skipped.
4. When the runs are done (or you have to stop), run the **last cell** and send `MyDrive/pinn/real5_m8_results.zip` to the project lead.

In [ ]:
LAYOUTS = [0, 1, 2]
DRIVE_ROOT = "/content/drive/MyDrive/pinn"

In [ ]:
import glob
import os
import shutil
import zipfile
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/runs/logs", exist_ok=True)
CODE = "/content/pinn"
shutil.rmtree(CODE, ignore_errors=True)
with zipfile.ZipFile(f"{DRIVE_ROOT}/pinn_code.zip") as z:
    z.extractall(CODE)
%cd {CODE}
assert os.path.exists("data/reference/paddy_share_district.csv"), "old pinn_code.zip: ask the project lead for the new one"

In [ ]:
# PyTorch is already installed on Colab; this adds everything else (about 1 minute).
!pip install -q -r requirements.txt
!pip install -q -e .
os.environ["PINN_RUNS_DIR"] = f"{DRIVE_ROOT}/runs"
!python scripts/check_env.py
!python -m pytest -q -m "not data"

## Runs
- `nophys`: no physics at all (the same network trained on the well data only)
- `noq`: physics on, but no pumping
- `ksweep`: 3 and 8 zones instead of 5

In [ ]:
def done(pattern):
    return bool(glob.glob(f"{DRIVE_ROOT}/runs/{pattern}/metrics.json"))

LOG = f"{DRIVE_ROOT}/runs/logs/real5_m8.log"
NOPHYS = ("--set training.stages.data_only=49000 --set training.stages.physics_frozen=0 "
          "--set training.stages.joint=0 --set training.stages.lbfgs=0")
RUNS = [("kmeansk5_shared", "nophys", "--k 5 " + NOPHYS),
        ("kmeansk5_shared", "noq", "--k 5 --set q_net.off=true"),
        ("kmeansk3_shared", "ksweep", "--k 3"),
        ("kmeansk8_shared", "ksweep", "--k 8")]
for layout in LAYOUTS:
    for name, tag, extra in RUNS:
        if done(f"*_real_L{layout}_{name}_s0_{tag}"):
            print("already done:", name, tag, layout)
            continue
        !python -u scripts/run_real.py --shared --layout {layout} --seed 0 --tag {tag} {extra} --stage-only val 2>&1 | tee -a {LOG}

## Bundle the results for Claude
Makes `MyDrive/pinn/real5_m8_results.zip` with every **finished** run (run it any time).
Download it, put it in the `pinn` folder on the laptop, then tell Claude.

In [ ]:
out = f"{DRIVE_ROOT}/real5_m8_results.zip"
n = 0
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for name, tag, _ in RUNS:
        for run in glob.glob(f"{DRIVE_ROOT}/runs/*_real_L*_{name}_s0_{tag}"):
            if not os.path.exists(f"{run}/metrics.json"):
                continue
            n += 1
            for path in glob.glob(f"{run}/*"):
                z.write(path, os.path.relpath(path, f"{DRIVE_ROOT}/runs"))
    if os.path.exists(LOG):
        z.write(LOG, "logs/real5_m8.log")
print("wrote", out, f"({n} finished runs, {os.path.getsize(out) / 1e6:.2f} MB)")